# Data/MC agreement summary over all BDT input variables

Reads `plots/all_p_value_info.pkl` written by `python src/plot_many_variables.py` (WC generic preselection, open-data prediction vs data).

- **No DetVar** / **With DetVar**: covariance without / with detector-variation systematics
- **Full χ²**: χ² over all bins, ndf = number of bins
- **Global decomp**: largest |ε_i| from the χ² eigen-decomposition, trials-corrected for the number of components, p_global = 1 − (1 − p_local)^N

The variables are strongly correlated, so their p-values are not independent and the uniform expectation is only a rough guide.

Plots are saved as PDF and PNG in `plots/bdt_var_p_values/`.

In [ ]:
import pickle
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import chi2, norm

plt.rcParams.update({"font.size": 12})

out_dir = Path("../plots/bdt_var_p_values")
out_dir.mkdir(parents=True, exist_ok=True)

def save(fig, name):
    fig.savefig(out_dir / f"{name}.pdf", bbox_inches="tight")
    fig.savefig(out_dir / f"{name}.png", bbox_inches="tight", dpi=200)

## Load, and recompute p-values and σ with full precision

The stored values use `1 - chi2.cdf(...)`, which underflows to p = 0 (σ = ∞) for large χ². They are recomputed from the stored χ²/ndf and max local σ using log survival functions. The convention is the same two-sided one, σ = Φ⁻¹(1 − p/2).

In [ ]:
with open("../plots/all_p_value_info.pkl", "rb") as f:
    df = pd.DataFrame(pickle.load(f))
print(f"{len(df)} variables")

def logp_to_sigma(logp):
    return norm.isf(np.exp(logp - np.log(2)))

for cov in ["nodetvar", "tot"]:
    logp = chi2.logsf(df[f"{cov}_chi2"], df["ndf"])
    df[f"{cov}_p"] = np.exp(logp)
    df[f"{cov}_sig"] = logp_to_sigma(logp)

    # global: max local |eps| (1 dof), trials-corrected over ndf components; log(1 - (1 - p)^N) computed stably
    local_logp = chi2.logsf(df[f"{cov}_max_local_sigma"] ** 2, 1)
    global_p = -np.expm1(df["ndf"] * np.log1p(-np.exp(local_logp)))
    global_logp = np.where(global_p < 1e-12, np.log(df["ndf"]) + local_logp, np.log(np.clip(global_p, 1e-300, None)))
    df[f"{cov}_global_p"] = np.exp(global_logp)
    df[f"{cov}_global_sig"] = logp_to_sigma(global_logp)

    # cross-check against the stored values where the old 1 - cdf is still precise
    precise = df[f"{cov}_p"] > 1e-10
    assert np.allclose(df.loc[precise, f"{cov}_sig"], df.loc[precise, f"{cov}_sigma"], rtol=1e-6, atol=1e-6)

tests = {
    "nodetvar": ("No DetVar, full $\\chi^2$", "tab:blue", "-"),
    "nodetvar_global": ("No DetVar, global decomp", "tab:blue", "--"),
    "tot": ("With DetVar, full $\\chi^2$", "tab:orange", "-"),
    "tot_global": ("With DetVar, global decomp", "tab:orange", "--"),
}

## p-value and σ distributions

Each plot is made twice: with all four tests, and with only the With DetVar curves (files ending in `_with_detvar_only`).

In [ ]:
with_detvar_tests = {k: v for k, v in tests.items() if k.startswith("tot")}

def plot_p_values(tests_to_plot, name):
    bins = np.linspace(0, 1, 26)
    fig, ax = plt.subplots(figsize=(10, 6))
    for key, (lab, color, ls) in tests_to_plot.items():
        ax.hist(df[f"{key}_p"], bins=bins, histtype="step", label=lab, color=color, linestyle=ls, lw=1.5)
    ax.axhline(len(df) / (len(bins) - 1), color="k", linestyle=":", label="Uniform expectation")
    ax.set_xlabel("p-value")
    ax.set_ylabel("Variables")
    ax.set_yscale("log")
    ax.set_title(f"All {len(df)} BDT variables, WC generic preselection")
    ax.legend(fontsize=10)
    save(fig, name)
    plt.show()

def plot_sigmas(tests_to_plot, name):
    max_sig = max(df[f"{key}_sig"].max() for key in tests_to_plot)
    bins = np.arange(0, np.ceil(max_sig) + 0.25, 0.25)
    fig, ax = plt.subplots(figsize=(10, 6))
    max_count = 0
    for key, (lab, color, ls) in tests_to_plot.items():
        counts, _, _ = ax.hist(df[f"{key}_sig"], bins=bins, histtype="step", label=lab, color=color, linestyle=ls, lw=1.5)
        max_count = max(max_count, counts.max())
    # expected for uniform p with the two-sided convention: sigma ~ |N(0,1)|
    expected_counts = len(df) * 2 * (norm.sf(bins[:-1]) - norm.sf(bins[1:]))
    ax.step(bins[:-1], expected_counts, where="post", color="k", linestyle=":", label="Expected for uniform p")
    ax.set_xlabel(r"$\sigma$")
    ax.set_ylabel("Variables")
    ax.set_yscale("log")
    ax.set_ylim(0.5, max_count * 3)
    ax.set_title(f"All {len(df)} BDT variables, WC generic preselection")
    ax.legend(fontsize=10)
    save(fig, name)
    plt.show()

In [ ]:
plot_p_values(tests, "p_value_distribution")
plot_p_values(with_detvar_tests, "p_value_distribution_with_detvar_only")

In [ ]:
plot_sigmas(tests, "sigma_distribution")
plot_sigmas(with_detvar_tests, "sigma_distribution_with_detvar_only")

## Most discrepant variables (With DetVar, full χ²)

In [ ]:
sigma_threshold = 2
discrepant = df[df["tot_sig"] > sigma_threshold].sort_values("tot_sig", ascending=False).reset_index(drop=True)
show_cols = ["var", "ndf", "tot_chi2", "tot_sig", "tot_global_sig", "nodetvar_chi2", "nodetvar_sig", "nodetvar_global_sig"]
print(f"{len(discrepant)} variables above {sigma_threshold} sigma (expected for uniform, independent p: {len(df) * 2 * norm.sf(sigma_threshold):.1f})")
pd.set_option("display.max_colwidth", 60)
discrepant[show_cols].round(2)

## Combined PDF of the discrepant variables, sorted by descending σ

Re-runs `src/plot_many_variables.py` on just the discrepant variables, in descending-σ order, with two extra columns on each page: the systematic and DetVar breakdowns, and each DetVar variation's spectrum with its (Var − CV)/CV fractional difference. The script runs as a separate process so that its large data loads stay out of this kernel. It writes:
- the individual pages to `plots/discrepant_bdt_vars/`
- the combined PDF to `plots/discrepant_bdt_vars.pdf`
- the p-value info to `plots/discrepant_bdt_vars_p_value_info.pkl`

It does not touch `all_p_value_info.pkl`. The systematic covariances come from the cache filled by the full run, so this is much faster than the full run.

Set `run_plots = False` to skip re-plotting and only check an existing output.

In [ ]:
import subprocess
import sys

run_plots = True
output_name = "discrepant_bdt_vars"

if run_plots:
    cmd = [sys.executable, "src/plot_many_variables.py",
           "--vars", ",".join(discrepant["var"]),
           "--output-name", output_name,
           "--sys-breakdown-panels",
           "--detvar-variation-panels",
           "--clear-prev"]
    proc = subprocess.Popen(cmd, cwd="..", stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    log_lines = []
    for line in proc.stdout:
        log_lines.append(line)
        # only echo progress lines, the full log is kept in log_lines
        if line.startswith(("plotting", "skipping", "Final PDF", "WARNING", "ERROR", "Traceback")):
            print(line, end="")
    if proc.wait() != 0:
        print("".join(log_lines[-50:]))
        raise RuntimeError(f"plot_many_variables.py failed with exit code {proc.returncode}")

In [ ]:
# the new run's chi2 values should reproduce the ones from the full run
with open(f"../plots/{output_name}_p_value_info.pkl", "rb") as f:
    rerun_df = pd.DataFrame(pickle.load(f))
assert rerun_df["var"].tolist() == discrepant["var"].tolist(), "page order differs from the sigma ordering"
check = discrepant[["var", "tot_chi2", "nodetvar_chi2"]].merge(rerun_df[["var", "tot_chi2", "nodetvar_chi2"]], on="var", suffixes=("", "_rerun"))
for cov in ["tot", "nodetvar"]:
    max_rel_diff = (np.abs(check[f"{cov}_chi2_rerun"] - check[f"{cov}_chi2"]) / check[f"{cov}_chi2"]).max()
    print(f"{cov} chi2: max relative difference vs full run = {max_rel_diff:.2e}")
print(f"{len(rerun_df)} pages in {Path(f'../plots/{output_name}.pdf').resolve()}")

## LaTeX section

Writes `temp_latex_bdt_var_p_values.tex` in the repository root. It contains the four distribution plots, a count table, a table of the discrepant variables, and one figure per discrepant-variable page. All numbers are filled in from this notebook.

The document needs `\\newcommand{\\bdtvarfigs}{<path to plots>}` plus the `graphicx`, `subcaption`, `float` and `booktabs` packages.

In [ ]:
def tex_var(var):
    return "\\texttt{" + var.replace("_", "\\_") + "}"

def tex_sigma(sig):
    return f"{sig:.2f}"

n_vars = len(df)
expected_above = {t: n_vars * 2 * norm.sf(t) for t in [2, 3, 4, 5]}
count_rows = []
for key, label in [("nodetvar", "No DetVar, full $\\chi^2$"), ("nodetvar_global", "No DetVar, global decomposition"),
                   ("tot", "With DetVar, full $\\chi^2$"), ("tot_global", "With DetVar, global decomposition")]:
    count_rows.append(label + " & " + " & ".join(str(int((df[f"{key}_sig"] > t).sum())) for t in expected_above) + " \\\\")
count_rows.append("\\midrule")
count_rows.append("Expected (uniform, independent $p$) & " + " & ".join(f"{v:.2g}" for v in expected_above.values()) + " \\\\")

table_rows = []
for rank, r in discrepant.iterrows():
    table_rows.append(
        f"{rank + 1} & {tex_var(r['var'])} & {int(r['ndf'])} & {r['tot_chi2']:.1f} & {tex_sigma(r['tot_sig'])} & {tex_sigma(r['tot_global_sig'])}"
        f" & {r['nodetvar_chi2']:.1f} & {tex_sigma(r['nodetvar_sig'])} & {tex_sigma(r['nodetvar_global_sig'])} \\\\"
    )

page_figures = []
for rank, r in discrepant.iterrows():
    page_figures.append(f"""\\begin{{figure}}[H]
    \\centering
    \\includegraphics[width=\\textwidth]{{\\bdtvarfigs/{output_name}/{r['var']}.pdf}}
    \\caption{{{tex_var(r['var'])} at the WC generic neutrino preselection (rank {rank + 1}): with DetVar, $\\chi^2/\\mathrm{{ndf}} = {r['tot_chi2']:.1f}/{int(r['ndf'])}$ ({tex_sigma(r['tot_sig'])}$\\sigma$, global decomposition {tex_sigma(r['tot_global_sig'])}$\\sigma$); without DetVar, $\\chi^2/\\mathrm{{ndf}} = {r['nodetvar_chi2']:.1f}/{int(r['ndf'])}$ ({tex_sigma(r['nodetvar_sig'])}$\\sigma$). Panel layout as described in Section~\\ref{{sec:bdt_var_discrepant}}.}}
    \\label{{fig:bdt_var_discrepant_{r['var']}}}
\\end{{figure}}""")

tex = f"""% Generated from ipynb_notebooks/plot_all_bdt_vars.ipynb; needs \\newcommand{{\\bdtvarfigs}}{{<path to plots>}}
% Requires \\usepackage{{graphicx, subcaption, float, booktabs}}
\\subsection{{Data/MC Agreement of the BDT Input Variables}}
\\label{{sec:bdt_var_agreement}}

All {n_vars} BDT input variables are compared between the Runs 1--5 open data and the prediction after the WC generic neutrino preselection.
For each variable the binning is chosen automatically, and the agreement is quantified with two covariance matrices: one including statistical uncertainties and the flux, cross section, and reinteraction systematics (``No DetVar''), and one also including detector-variation systematics (``With DetVar'').
Two test statistics are computed for each.
The full $\\chi^2$ uses all bins, with the number of degrees of freedom equal to the number of bins with more than one data event.
The global decomposition test decomposes the $\\chi^2$ into independent components $\\epsilon_i$ using the covariance matrix, converts the largest $|\\epsilon_i|$ into a local $p$-value, and corrects it for the number of components $N$ as $p_\\mathrm{{global}} = 1 - (1 - p_\\mathrm{{local}})^N$.
$p$-values are converted to significances with the two-sided convention $\\sigma = \\Phi^{{-1}}(1 - p/2)$.
Because the variables are strongly correlated with each other, their $p$-values are not independent, and the expectation for uniform $p$-values is only a rough reference.

Figure~\\ref{{fig:bdt_var_p_values}} shows the distributions of the $p$-values and significances, and Table~\\ref{{tab:bdt_var_sigma_counts}} counts the variables above several significance thresholds.
Without detector systematics many variables disagree; with them, the $p$-values pile up near one, and fewer variables exceed each threshold than expected for uniform $p$-values, suggesting the detector-variation uncertainties are conservative for most variables.

\\begin{{figure}}[H]
    \\centering
    \\begin{{subfigure}}[b]{{0.49\\textwidth}}
        \\includegraphics[width=\\textwidth]{{\\bdtvarfigs/bdt_var_p_values/p_value_distribution.pdf}}
        \\caption{{}}
    \\end{{subfigure}}
    \\begin{{subfigure}}[b]{{0.49\\textwidth}}
        \\includegraphics[width=\\textwidth]{{\\bdtvarfigs/bdt_var_p_values/p_value_distribution_with_detvar_only.pdf}}
        \\caption{{}}
    \\end{{subfigure}}
    \\begin{{subfigure}}[b]{{0.49\\textwidth}}
        \\includegraphics[width=\\textwidth]{{\\bdtvarfigs/bdt_var_p_values/sigma_distribution.pdf}}
        \\caption{{}}
    \\end{{subfigure}}
    \\begin{{subfigure}}[b]{{0.49\\textwidth}}
        \\includegraphics[width=\\textwidth]{{\\bdtvarfigs/bdt_var_p_values/sigma_distribution_with_detvar_only.pdf}}
        \\caption{{}}
    \\end{{subfigure}}
    \\caption{{Data/MC agreement of the {n_vars} BDT input variables after the WC generic neutrino preselection: (a) $p$-values for all four tests, (b) $p$-values with DetVar systematics only, (c) significances for all four tests, (d) significances with DetVar systematics only. Dotted black lines show the expectation for uniform, independent $p$-values.}}
    \\label{{fig:bdt_var_p_values}}
\\end{{figure}}

\\begin{{table}}[H]
\\centering
\\caption{{Number of the {n_vars} BDT input variables above each significance threshold, compared with the expectation for uniform, independent $p$-values.}}
\\label{{tab:bdt_var_sigma_counts}}
\\begin{{tabular}}{{lrrrr}}
\\toprule
Test & $>2\\sigma$ & $>3\\sigma$ & $>4\\sigma$ & $>5\\sigma$ \\\\
\\midrule
{chr(10).join(count_rows)}
\\bottomrule
\\end{{tabular}}
\\end{{table}}

\\subsubsection{{Most Discrepant Variables}}
\\label{{sec:bdt_var_discrepant}}

Table~\\ref{{tab:bdt_var_discrepant}} lists the {len(discrepant)} variables whose full $\\chi^2$ with DetVar systematics exceeds {sigma_threshold}$\\sigma$, compared with about {expected_above[sigma_threshold]:.0f} expected for uniform, independent $p$-values.
Figures~\\ref{{fig:bdt_var_discrepant_{discrepant['var'].iloc[0]}}}--\\ref{{fig:bdt_var_discrepant_{discrepant['var'].iloc[-1]}}} show each of them, in decreasing order of significance.
Each figure has three columns.
The left column shows the data and the stacked prediction with the No-DetVar (``/'' hatching) and total (``\\textbackslash'' hatching) systematic uncertainty bands, the data/prediction ratio, and the $\\chi^2$ decomposition components $\\epsilon_i$.
The middle column shows the fractional uncertainty broken down by source (top) and by detector variation (bottom).
The right column shows the spectrum of each detector variation compared with the CV, and the fractional difference $(\\mathrm{{Var}} - \\mathrm{{CV}})/\\mathrm{{CV}}$, where each variation is compared only with the CV events it is matched to.
Shaded bins at the edges of a plot hold sentinel default values, such as $-2147483648$ (the minimum 32-bit integer), which typically mean the quantity is undefined for the event.

\\begin{{table}}[H]
\\centering
\\small
\\caption{{BDT input variables whose full $\\chi^2$ with DetVar systematics exceeds {sigma_threshold}$\\sigma$ after the WC generic neutrino preselection, sorted by that significance. ``Global'' is the trials-corrected significance of the largest $\\chi^2$ decomposition component.}}
\\label{{tab:bdt_var_discrepant}}
\\resizebox{{\\textwidth}}{{!}}{{%
\\begin{{tabular}}{{rlrrrrrrr}}
\\toprule
& & & \\multicolumn{{3}}{{c}}{{With DetVar}} & \\multicolumn{{3}}{{c}}{{No DetVar}} \\\\
\\cmidrule(lr){{4-6}} \\cmidrule(lr){{7-9}}
Rank & Variable & ndf & $\\chi^2$ & $\\sigma$ & Global $\\sigma$ & $\\chi^2$ & $\\sigma$ & Global $\\sigma$ \\\\
\\midrule
{chr(10).join(table_rows)}
\\bottomrule
\\end{{tabular}}%
}}
\\end{{table}}

{(chr(10) + chr(10)).join(page_figures)}
"""

tex_path = Path("../temp_latex_bdt_var_p_values.tex")
tex_path.write_text(tex)
print(f"wrote {tex_path.resolve()}")